# 完全自包含：Excel 全量训练与安全推荐测试

本 notebook 每次只从 Excel 重新训练，不读取已有模型、报告、源码或其他 notebook。

## 配置与依赖检查

In [ ]:
from pathlib import Path
import importlib.util
required = ['numpy', 'pandas', 'scipy', 'sklearn', 'joblib', 'openpyxl', 'lightgbm', 'catboost']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f'缺少运行依赖: {missing}')
EXCEL_NAME = '4_month_data_2026_02_01_2026_06_25.xlsx'
search_roots = [Path.cwd(), *Path.cwd().parents]
EXCEL_PATH = next((root / EXCEL_NAME for root in search_roots if (root / EXCEL_NAME).exists()), None)
if EXCEL_PATH is None:
    raise FileNotFoundError(f'未找到 Excel，请把 {EXCEL_NAME} 放在 notebook 工作目录或其父目录。')
PROJECT_ROOT = EXCEL_PATH.parent
OUTPUT_ROOT = PROJECT_ROOT / 'advanced_furnace_ml/complete_notebook_output'
ARTIFACTS_DIR = OUTPUT_ROOT / 'artifacts'
REPORTS_DIR = OUTPUT_ROOT / 'reports'
for directory in (OUTPUT_ROOT, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print({'excel': str(EXCEL_PATH), 'isolated_output': str(OUTPUT_ROOT), 'missing_dependencies': missing})

In [ ]:
# 运行时代码已作为字符串快照嵌入本 notebook；执行时不读取项目源码。
RUNTIME_SOURCES = {'__init__.py': '"""Advanced, isolated furnace gas prediction and recommendation experiments."""\n'
                '\n'
                '__version__ = "0.1.0"\n',
 'artifacts.py': '"""Persist, validate, predict, and recommend from the advanced bundle."""\n'
                 '\n'
                 'from pathlib import Path\n'
                 '\n'
                 'import joblib\n'
                 'import numpy as np\n'
                 'import pandas as pd\n'
                 '\n'
                 'from .data import FEATURE_COLS\n'
                 'from .optimization import bayesian_search, genetic_search, random_search\n'
                 'from .recommendation import build_recommendation_context, recommend_or_fallback\n'
                 'from .uncertainty import prediction_interval\n'
                 '\n'
                 '\n'
                 'def save_bundle(bundle: dict, path: str | Path):\n'
                 '    destination = Path(path)\n'
                 '    if destination.suffix != ".joblib":\n'
                 '        raise ValueError("模型产物必须使用 .joblib 扩展名。")\n'
                 '    destination.parent.mkdir(parents=True, exist_ok=True)\n'
                 '    joblib.dump(bundle, destination)\n'
                 '    return destination\n'
                 '\n'
                 '\n'
                 'def load_bundle(path: str | Path):\n'
                 '    bundle = joblib.load(path)\n'
                 '    if not isinstance(bundle, dict) or bundle.get("artifact_version") != "advanced-furnace-1.0":\n'
                 '        raise ValueError("不支持的高级熔炼炉模型包。")\n'
                 '    return bundle\n'
                 '\n'
                 '\n'
                 'def validate_frame(frame: pd.DataFrame):\n'
                 '    missing = [column for column in FEATURE_COLS if column not in frame]\n'
                 '    extra = [column for column in frame if column not in FEATURE_COLS]\n'
                 '    if missing:\n'
                 '        raise ValueError(f"输入缺少特征: {missing}")\n'
                 '    if extra:\n'
                 '        raise ValueError(f"输入包含额外特征: {extra}")\n'
                 '    numeric = frame[FEATURE_COLS].apply(pd.to_numeric, errors="coerce")\n'
                 '    invalid = [column for column in FEATURE_COLS if numeric[column].isna().any()]\n'
                 '    if invalid:\n'
                 '        raise ValueError(f"输入必须为完整数值: {invalid}")\n'
                 '    return numeric\n'
                 '\n'
                 '\n'
                 'def predict_bundle(bundle: dict, frame: pd.DataFrame):\n'
                 '    X = validate_frame(frame)\n'
                 '    prediction = np.asarray(bundle["prediction_model"].predict(X), dtype=float)\n'
                 '    low, high = prediction_interval(prediction, bundle["conformal_radius_90"])\n'
                 '    fold_matrix = np.vstack([model.predict(X) for model in bundle["prediction_fold_models"]])\n'
                 '    return pd.DataFrame({\n'
                 '        "predicted_gas": prediction,\n'
                 '        "prediction_lower_90": low,\n'
                 '        "prediction_upper_90": high,\n'
                 '        "chronological_fold_std": fold_matrix.std(axis=0),\n'
                 '        "model_name": bundle["selected_model_name"],\n'
                 '    })\n'
                 '\n'
                 '\n'
                 'def recommend_bundle(bundle: dict, total_weight: float, budget: int = 600, seed: int = 42):\n'
                 '    context = build_recommendation_context(\n'
                 '        bundle["training_data"], total_weight, bundle["recommendation_trust_ratio"]\n'
                 '    )\n'
                 '    functions = {\n'
                 '        "random_search": random_search,\n'
                 '        "genetic_algorithm": genetic_search,\n'
                 '        "bayesian_optimization": bayesian_search,\n'
                 '    }\n'
                 '    function = functions[bundle["recommendation_optimizer"]]\n'
                 '    scored = function(\n'
                 '        bundle["recommendation_model"],\n'
                 '        bundle["recommendation_fold_models"],\n'
                 '        context,\n'
                 '        bundle["feasibility_reference"],\n'
                 '        budget=budget,\n'
                 '        seed=seed,\n'
                 '    )\n'
                 '    result = recommend_or_fallback(scored, context)\n'
                 '    result["recommendation_model_name"] = bundle["recommendation_model_name"]\n'
                 '    result["optimizer"] = bundle["recommendation_optimizer"]\n'
                 '    return result\n',
 'data.py': '"""Chronological Excel loading and immutable data contracts."""\n'
            '\n'
            'from pathlib import Path\n'
            '\n'
            'import pandas as pd\n'
            '\n'
            '\n'
            'TARGET_COL = "熔炼炉B当前批次总气耗_PLC"\n'
            'FEATURE_COLS = [\n'
            '    "10#熔炼炉总投料重量(kg)",\n'
            '    "10#熔炼炉固体料重量比例",\n'
            '    "熔炼炉B当前批次熔炼时间_PLC",\n'
            '    "熔炼炉B当前批次等待时长_PLC",\n'
            '    "熔炼炉B当前批次炉门打开次数_PLC",\n'
            '    "熔炼炉B当前批次炉门打开时长_PLC",\n'
            ']\n'
            'WEIGHT_COL, SOLID_COL, MELTING_COL, WAIT_COL, DOOR_COUNT_COL, DOOR_DURATION_COL = FEATURE_COLS\n'
            '\n'
            '\n'
            'def load_batch_data(path: str | Path, sheet_name: str = "Sheet1") -> pd.DataFrame:\n'
            '    raw = pd.read_excel(path, sheet_name=sheet_name, header=None, engine="openpyxl")\n'
            '    batch_cols = [\n'
            '        col for col in raw.columns\n'
            '        if isinstance(raw.iloc[0, col], str) and raw.iloc[0, col].startswith("ER")\n'
            '    ]\n'
            '    variable_rows = [\n'
            '        row for row in raw.index\n'
            '        if isinstance(raw.iloc[row, 0], str) and raw.iloc[row, 0] != "Grand Total"\n'
            '    ]\n'
            '    values = raw.loc[variable_rows, batch_cols].apply(pd.to_numeric, errors="coerce").T\n'
            '    values.columns = raw.loc[variable_rows, 0].astype(str).tolist()\n'
            '    values = values.reset_index(drop=True)\n'
            '    values.insert(0, "batch_id", [str(raw.iloc[0, col]).strip() for col in batch_cols])\n'
            '    required = FEATURE_COLS + [TARGET_COL]\n'
            '    missing = [column for column in required if column not in values]\n'
            '    if missing:\n'
            '        raise ValueError(f"Excel 缺少建模列: {missing}")\n'
            '    if values["batch_id"].duplicated().any():\n'
            '        raise ValueError("Excel 包含重复炉次号。")\n'
            '    result = values[["batch_id"] + required].copy()\n'
            '    if result[TARGET_COL].isna().any():\n'
            '        raise ValueError("目标总气耗包含缺失值。")\n'
            '    return result\n'
            '\n'
            '\n'
            'def mark_target_outliers(df: pd.DataFrame) -> pd.DataFrame:\n'
            '    result = df.copy()\n'
            '    q1, q3 = result[TARGET_COL].quantile([0.25, 0.75])\n'
            '    result["is_high_gas_outlier"] = result[TARGET_COL] > q3 + 1.5 * (q3 - q1)\n'
            '    return result\n'
            '\n'
            '\n'
            'def chronological_dev_lock_split(df: pd.DataFrame, lock_size: int = 44) -> tuple[pd.DataFrame, '
            'pd.DataFrame]:\n'
            '    if len(df) <= lock_size:\n'
            '        raise ValueError("数据量不足以建立锁定测试集。")\n'
            '    split = len(df) - lock_size\n'
            '    return df.iloc[:split].copy(), df.iloc[split:].copy()\n',
 'experiment.py': '"""Development-only model selection and one-time locked audit."""\n'
                  '\n'
                  'from dataclasses import dataclass, field\n'
                  'import warnings\n'
                  '\n'
                  'import numpy as np\n'
                  'import pandas as pd\n'
                  'from sklearn.base import clone\n'
                  'from sklearn.exceptions import ConvergenceWarning\n'
                  'from sklearn.model_selection import ShuffleSplit\n'
                  '\n'
                  'from .data import FEATURE_COLS, TARGET_COL\n'
                  'from .models import (\n'
                  '    ResidualBoostRegressor,\n'
                  '    RouteRegressor,\n'
                  '    WeightedOOFEnsemble,\n'
                  '    build_base_models,\n'
                  '    build_tree_model_variants,\n'
                  '    fit_nonnegative_ensemble_weights,\n'
                  ')\n'
                  'from .uncertainty import conformal_radius, prediction_interval\n'
                  'from .validation import bootstrap_metric_interval, regression_metrics\n'
                  '\n'
                  '\n'
                  '@dataclass\n'
                  'class CandidateResult:\n'
                  '    name: str\n'
                  '    route: str\n'
                  '    estimator_template: object\n'
                  '    oof_predictions: np.ndarray\n'
                  '    fold_models: list\n'
                  '    fold_metrics: pd.DataFrame\n'
                  '    selection_score: float\n'
                  '    conformal_radius_90: float\n'
                  '\n'
                  '\n'
                  '@dataclass\n'
                  'class ModelExperiment:\n'
                  '    dev_df: pd.DataFrame\n'
                  '    results: dict[str, CandidateResult]\n'
                  '    summary: pd.DataFrame\n'
                  '    selected_name: str\n'
                  '    ensemble_members: list[str]\n'
                  '    ensemble_weights: np.ndarray\n'
                  '    tree_tuning_summary: pd.DataFrame\n'
                  '    frozen: bool = False\n'
                  '    full_dev_models: dict[str, object] = field(default_factory=dict)\n'
                  '\n'
                  '    def freeze(self):\n'
                  '        self.frozen = True\n'
                  '\n'
                  '\n'
                  'def evaluate_candidate(name, estimator, route, dev_df, splits) -> CandidateResult:\n'
                  '    X = dev_df[FEATURE_COLS]\n'
                  '    y = dev_df[TARGET_COL].to_numpy(dtype=float)\n'
                  '    oof = np.full(len(dev_df), np.nan)\n'
                  '    models = []\n'
                  '    rows = []\n'
                  '    for fold, (train_idx, valid_idx) in enumerate(splits):\n'
                  '        model = clone(estimator)\n'
                  '        with warnings.catch_warnings():\n'
                  '            warnings.simplefilter("ignore", ConvergenceWarning)\n'
                  '            model.fit(X.iloc[train_idx], y[train_idx])\n'
                  '        prediction = model.predict(X.iloc[valid_idx])\n'
                  '        oof[valid_idx] = prediction\n'
                  '        models.append(model)\n'
                  '        rows.append({\n'
                  '            "candidate": name,\n'
                  '            "route": route,\n'
                  '            "fold": fold,\n'
                  '            "train_size": len(train_idx),\n'
                  '            "valid_size": len(valid_idx),\n'
                  '            **regression_metrics(y[valid_idx], prediction),\n'
                  '        })\n'
                  '    metrics = pd.DataFrame(rows)\n'
                  '    score = float(metrics["rmse"].mean() + 0.25 * metrics["rmse"].std())\n'
                  '    return CandidateResult(\n'
                  '        name=name,\n'
                  '        route=route,\n'
                  '        estimator_template=estimator,\n'
                  '        oof_predictions=oof,\n'
                  '        fold_models=models,\n'
                  '        fold_metrics=metrics,\n'
                  '        selection_score=score,\n'
                  '        conformal_radius_90=conformal_radius(y, oof, 0.90),\n'
                  '    )\n'
                  '\n'
                  '\n'
                  'def _summary_row(result: CandidateResult):\n'
                  '    metrics = result.fold_metrics\n'
                  '    return {\n'
                  '        "candidate": result.name,\n'
                  '        "route": result.route,\n'
                  '        "mae_mean": metrics["mae"].mean(),\n'
                  '        "rmse_mean": metrics["rmse"].mean(),\n'
                  '        "rmse_std": metrics["rmse"].std(),\n'
                  '        "rmse_worst": metrics["rmse"].max(),\n'
                  '        "wape_mean": metrics["wape"].mean(),\n'
                  '        "r2_mean": metrics["r2"].mean(),\n'
                  '        "error_gt_10pct_rate": metrics["error_gt_10pct_rate"].mean(),\n'
                  '        "selection_score": result.selection_score,\n'
                  '        "conformal_radius_90": result.conformal_radius_90,\n'
                  '    }\n'
                  '\n'
                  '\n'
                  'def run_model_matrix(dev_df, splits, model_names=None, routes=("direct", "unit"), tune_trees: bool '
                  '= True) -> ModelExperiment:\n'
                  '    base_models = build_base_models()\n'
                  '    names = tuple(base_models) if model_names is None else tuple(model_names)\n'
                  '    tuning_rows = []\n'
                  '    if tune_trees:\n'
                  '        variants = build_tree_model_variants()\n'
                  '        for tree_name in ("LightGBM", "CatBoost"):\n'
                  '            if tree_name not in names:\n'
                  '                continue\n'
                  '            scored_variants = []\n'
                  '            for label, template in variants[tree_name].items():\n'
                  '                variant_result = evaluate_candidate(\n'
                  '                    f"tuning__{tree_name}__{label}",\n'
                  '                    RouteRegressor(template, "direct"),\n'
                  '                    "direct",\n'
                  '                    dev_df,\n'
                  '                    splits,\n'
                  '                )\n'
                  '                scored_variants.append((variant_result.selection_score, label, template))\n'
                  '                tuning_rows.append({\n'
                  '                    "model": tree_name,\n'
                  '                    "variant": label,\n'
                  '                    "selection_score": variant_result.selection_score,\n'
                  '                    "rmse_mean": variant_result.fold_metrics["rmse"].mean(),\n'
                  '                    "rmse_std": variant_result.fold_metrics["rmse"].std(),\n'
                  '                })\n'
                  '            _, selected_label, selected_template = min(scored_variants, key=lambda item: item[0])\n'
                  '            base_models[tree_name] = selected_template\n'
                  '            for row in tuning_rows:\n'
                  '                if row["model"] == tree_name:\n'
                  '                    row["selected"] = row["variant"] == selected_label\n'
                  '        if "LightGBM" in names:\n'
                  '            base_models["Ridge+LGBMResidual"] = ResidualBoostRegressor(\n'
                  '                base_models["Ridge"], base_models["LightGBM"]\n'
                  '            )\n'
                  '            base_models["Huber+LGBMResidual"] = ResidualBoostRegressor(\n'
                  '                base_models["Huber"], base_models["LightGBM"]\n'
                  '            )\n'
                  '    results = {}\n'
                  '    for model_name in names:\n'
                  '        for route in routes:\n'
                  '            candidate_name = f"{model_name}__{route}"\n'
                  '            estimator = RouteRegressor(base_models[model_name], route)\n'
                  '            results[candidate_name] = evaluate_candidate(candidate_name, estimator, route, dev_df, '
                  'splits)\n'
                  '\n'
                  '    ranked = sorted(results.values(), key=lambda item: item.selection_score)\n'
                  '    ensemble_sources = ranked[: min(4, len(ranked))]\n'
                  '    valid = np.logical_and.reduce([np.isfinite(item.oof_predictions) for item in '
                  'ensemble_sources])\n'
                  '    matrix = np.column_stack([item.oof_predictions[valid] for item in ensemble_sources])\n'
                  '    y_valid = dev_df[TARGET_COL].to_numpy(dtype=float)[valid]\n'
                  '    weights = fit_nonnegative_ensemble_weights(matrix, y_valid)\n'
                  '    ensemble = WeightedOOFEnsemble(\n'
                  '        [item.estimator_template for item in ensemble_sources], weights\n'
                  '    )\n'
                  '    results["OOFEnsemble"] = evaluate_candidate(\n'
                  '        "OOFEnsemble", ensemble, "mixed_total", dev_df, splits\n'
                  '    )\n'
                  '    summary = pd.DataFrame([_summary_row(result) for result in results.values()])\n'
                  '    summary = summary.sort_values("selection_score").reset_index(drop=True)\n'
                  '    return ModelExperiment(\n'
                  '        dev_df=dev_df.copy(),\n'
                  '        results=results,\n'
                  '        summary=summary,\n'
                  '        selected_name=str(summary.iloc[0]["candidate"]),\n'
                  '        ensemble_members=[item.name for item in ensemble_sources],\n'
                  '        ensemble_weights=weights,\n'
                  '        tree_tuning_summary=pd.DataFrame(tuning_rows),\n'
                  '    )\n'
                  '\n'
                  '\n'
                  'def random_cv_reference(experiment: ModelExperiment, n_splits: int = 3, seed: int = 42):\n'
                  '    X = experiment.dev_df[FEATURE_COLS]\n'
                  '    y = experiment.dev_df[TARGET_COL].to_numpy(dtype=float)\n'
                  '    splitter = ShuffleSplit(n_splits=n_splits, test_size=.20, random_state=seed)\n'
                  '    rows = []\n'
                  '    for name, result in experiment.results.items():\n'
                  '        for split_id, (train_idx, valid_idx) in enumerate(splitter.split(X)):\n'
                  '            model = clone(result.estimator_template)\n'
                  '            with warnings.catch_warnings():\n'
                  '                warnings.simplefilter("ignore", ConvergenceWarning)\n'
                  '                model.fit(X.iloc[train_idx], y[train_idx])\n'
                  '            prediction = model.predict(X.iloc[valid_idx])\n'
                  '            rows.append({\n'
                  '                "candidate": name,\n'
                  '                "split": split_id,\n'
                  '                "train_size": len(train_idx),\n'
                  '                "valid_size": len(valid_idx),\n'
                  '                **regression_metrics(y[valid_idx], prediction),\n'
                  '            })\n'
                  '    return pd.DataFrame(rows)\n'
                  '\n'
                  '\n'
                  'def audit_frozen_models(experiment: ModelExperiment, locked_df: pd.DataFrame) -> pd.DataFrame:\n'
                  '    if not experiment.frozen:\n'
                  '        raise RuntimeError("Model experiment must freeze() before locked audit.")\n'
                  '    X_dev = experiment.dev_df[FEATURE_COLS]\n'
                  '    y_dev = experiment.dev_df[TARGET_COL]\n'
                  '    X_lock = locked_df[FEATURE_COLS]\n'
                  '    y_lock = locked_df[TARGET_COL].to_numpy(dtype=float)\n'
                  '    rows = []\n'
                  '    for name, result in experiment.results.items():\n'
                  '        model = clone(result.estimator_template)\n'
                  '        with warnings.catch_warnings():\n'
                  '            warnings.simplefilter("ignore", ConvergenceWarning)\n'
                  '            model.fit(X_dev, y_dev)\n'
                  '        prediction = model.predict(X_lock)\n'
                  '        try:\n'
                  '            native_std_mean = float(np.mean(model.predict_std(X_lock)))\n'
                  '        except (AttributeError, TypeError):\n'
                  '            native_std_mean = np.nan\n'
                  '        experiment.full_dev_models[name] = model\n'
                  '        low, high = prediction_interval(prediction, result.conformal_radius_90)\n'
                  '        metrics = regression_metrics(y_lock, prediction)\n'
                  '        rmse_low, rmse_high = bootstrap_metric_interval(y_lock, prediction, "rmse", seed=42, '
                  'n_boot=500)\n'
                  '        rows.append({\n'
                  '            "candidate": name,\n'
                  '            "route": result.route,\n'
                  '            **metrics,\n'
                  '            "rmse_bootstrap_low95": rmse_low,\n'
                  '            "rmse_bootstrap_high95": rmse_high,\n'
                  '            "interval_coverage_90": float(((y_lock >= low) & (y_lock <= high)).mean()),\n'
                  '            "interval_mean_width": float(np.mean(high - low)),\n'
                  '            "native_model_std_mean": native_std_mean,\n'
                  '            "selected_before_lock": name == experiment.selected_name,\n'
                  '        })\n'
                  '    return pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)\n',
 'features.py': '"""Leakage-safe furnace feature engineering and target routes."""\n'
                '\n'
                'import numpy as np\n'
                'import pandas as pd\n'
                'from sklearn.base import BaseEstimator, TransformerMixin\n'
                '\n'
                'from .data import (\n'
                '    DOOR_COUNT_COL,\n'
                '    DOOR_DURATION_COL,\n'
                '    FEATURE_COLS,\n'
                '    MELTING_COL,\n'
                '    SOLID_COL,\n'
                '    TARGET_COL,\n'
                '    WAIT_COL,\n'
                '    WEIGHT_COL,\n'
                ')\n'
                '\n'
                '\n'
                'DERIVED_COLS = [\n'
                '    "derived__solid_weight_kg",\n'
                '    "derived__avg_door_duration",\n'
                '    "derived__door_count_per_melting_time",\n'
                '    "derived__door_duration_per_melting_time",\n'
                '    "derived__waiting_ratio",\n'
                '    "derived__non_waiting_melting_time",\n'
                ']\n'
                '\n'
                '\n'
                'class FurnaceFeatureEngineer(BaseEstimator, TransformerMixin):\n'
                '    def fit(self, X, y=None):\n'
                '        self.feature_names_in_ = np.asarray(FEATURE_COLS, dtype=object)\n'
                '        return self\n'
                '\n'
                '    def transform(self, X):\n'
                '        frame = pd.DataFrame(X).copy()[FEATURE_COLS].apply(pd.to_numeric, errors="coerce")\n'
                '        safe_melting = frame[MELTING_COL].replace(0, np.nan)\n'
                '        safe_count = frame[DOOR_COUNT_COL].replace(0, np.nan)\n'
                '        derived = pd.DataFrame(index=frame.index)\n'
                '        derived[DERIVED_COLS[0]] = frame[WEIGHT_COL] * frame[SOLID_COL] / 100.0\n'
                '        derived[DERIVED_COLS[1]] = frame[DOOR_DURATION_COL] / safe_count\n'
                '        derived[DERIVED_COLS[2]] = frame[DOOR_COUNT_COL] / safe_melting\n'
                '        derived[DERIVED_COLS[3]] = frame[DOOR_DURATION_COL] / safe_melting\n'
                '        derived[DERIVED_COLS[4]] = frame[WAIT_COL] / safe_melting\n'
                '        derived[DERIVED_COLS[5]] = frame[MELTING_COL] - frame[WAIT_COL]\n'
                '        derived = derived.replace([np.inf, -np.inf], np.nan)\n'
                '        return pd.concat([frame, derived], axis=1)\n'
                '\n'
                '    def get_feature_names_out(self, input_features=None):\n'
                '        return np.asarray(FEATURE_COLS + DERIVED_COLS, dtype=object)\n'
                '\n'
                '\n'
                'def _weight_tonnes(frame: pd.DataFrame, median: float | None = None) -> pd.Series:\n'
                '    weight = pd.to_numeric(frame[WEIGHT_COL], errors="coerce")\n'
                '    fill = float(weight.median()) if median is None else float(median)\n'
                '    return weight.fillna(fill) / 1000.0\n'
                '\n'
                '\n'
                'def target_for_route(df: pd.DataFrame, route: str) -> pd.Series:\n'
                '    target = pd.to_numeric(df[TARGET_COL], errors="coerce")\n'
                '    if route == "direct":\n'
                '        return target\n'
                '    if route == "unit":\n'
                '        return target / _weight_tonnes(df)\n'
                '    raise ValueError(f"未知目标路线: {route}")\n'
                '\n'
                '\n'
                'def prediction_to_total_gas(prediction, X: pd.DataFrame, route: str, weight_median: float | None = '
                'None) -> np.ndarray:\n'
                '    values = np.asarray(prediction, dtype=float)\n'
                '    if route == "direct":\n'
                '        return values\n'
                '    if route == "unit":\n'
                '        return values * _weight_tonnes(X, weight_median).to_numpy()\n'
                '    raise ValueError(f"未知目标路线: {route}")\n',
 'models.py': '"""Small-data candidate regressors, target routes, residuals, and ensembles."""\n'
              '\n'
              'from copy import deepcopy\n'
              '\n'
              'import numpy as np\n'
              'import pandas as pd\n'
              'from catboost import CatBoostRegressor\n'
              'from lightgbm import LGBMRegressor\n'
              'from scipy.optimize import minimize\n'
              'from sklearn.base import BaseEstimator, RegressorMixin, clone\n'
              'from sklearn.gaussian_process import GaussianProcessRegressor\n'
              'from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel\n'
              'from sklearn.impute import SimpleImputer\n'
              'from sklearn.linear_model import ElasticNet, HuberRegressor, Ridge\n'
              'from sklearn.pipeline import Pipeline\n'
              'from sklearn.preprocessing import RobustScaler, SplineTransformer, StandardScaler\n'
              '\n'
              'from .data import FEATURE_COLS, WEIGHT_COL\n'
              'from .features import FurnaceFeatureEngineer, prediction_to_total_gas\n'
              '\n'
              '\n'
              'def _imputer():\n'
              '    return SimpleImputer(strategy="median").set_output(transform="pandas")\n'
              '\n'
              '\n'
              'def _numeric_pipeline(model, scale="standard"):\n'
              '    steps = [("features", FurnaceFeatureEngineer()), ("imputer", _imputer())]\n'
              '    if scale == "standard":\n'
              '        steps.append(("scaler", StandardScaler()))\n'
              '    elif scale == "robust":\n'
              '        steps.append(("scaler", RobustScaler()))\n'
              '    steps.append(("model", model))\n'
              '    return Pipeline(steps)\n'
              '\n'
              '\n'
              'class ResidualBoostRegressor(BaseEstimator, RegressorMixin):\n'
              '    def __init__(self, base_model, residual_model):\n'
              '        self.base_model = base_model\n'
              '        self.residual_model = residual_model\n'
              '\n'
              '    def fit(self, X, y):\n'
              '        self.base_model_ = clone(self.base_model).fit(X, y)\n'
              '        residual = np.asarray(y, dtype=float) - self.base_model_.predict(X)\n'
              '        self.residual_model_ = clone(self.residual_model).fit(X, residual)\n'
              '        return self\n'
              '\n'
              '    def predict(self, X):\n'
              '        return self.base_model_.predict(X) + self.residual_model_.predict(X)\n'
              '\n'
              '\n'
              'class RouteRegressor(BaseEstimator, RegressorMixin):\n'
              '    def __init__(self, model, route="direct"):\n'
              '        self.model = model\n'
              '        self.route = route\n'
              '\n'
              '    def fit(self, X, y):\n'
              '        frame = pd.DataFrame(X).copy()\n'
              '        self.weight_median_ = float(pd.to_numeric(frame[WEIGHT_COL], errors="coerce").median())\n'
              '        target = np.asarray(y, dtype=float)\n'
              '        if self.route == "unit":\n'
              '            tonnes = pd.to_numeric(frame[WEIGHT_COL], '
              'errors="coerce").fillna(self.weight_median_).to_numpy() / 1000.0\n'
              '            target = target / tonnes\n'
              '        elif self.route != "direct":\n'
              '            raise ValueError(f"未知目标路线: {self.route}")\n'
              '        self.model_ = clone(self.model).fit(frame[FEATURE_COLS], target)\n'
              '        return self\n'
              '\n'
              '    def predict(self, X):\n'
              '        frame = pd.DataFrame(X).copy()[FEATURE_COLS]\n'
              '        prediction = self.model_.predict(frame)\n'
              '        return prediction_to_total_gas(prediction, frame, self.route, self.weight_median_)\n'
              '\n'
              '    def predict_std(self, X):\n'
              '        frame = pd.DataFrame(X).copy()[FEATURE_COLS]\n'
              '        if not isinstance(self.model_, Pipeline):\n'
              '            raise TypeError("底层模型不支持原生预测标准差。")\n'
              '        transformed = self.model_[:-1].transform(frame)\n'
              '        final_model = self.model_.steps[-1][1]\n'
              '        if not isinstance(final_model, GaussianProcessRegressor):\n'
              '            raise TypeError("底层模型不支持原生预测标准差。")\n'
              '        _, standard_deviation = final_model.predict(transformed, return_std=True)\n'
              '        return prediction_to_total_gas(\n'
              '            standard_deviation, frame, self.route, self.weight_median_\n'
              '        )\n'
              '\n'
              '\n'
              'class WeightedOOFEnsemble(BaseEstimator, RegressorMixin):\n'
              '    def __init__(self, models, weights):\n'
              '        self.models = models\n'
              '        self.weights = weights\n'
              '\n'
              '    def fit(self, X, y):\n'
              '        self.models_ = [clone(model).fit(X, y) for model in self.models]\n'
              '        return self\n'
              '\n'
              '    def predict(self, X):\n'
              '        members = getattr(self, "models_", self.models)\n'
              '        matrix = np.column_stack([model.predict(X) for model in members])\n'
              '        return matrix @ np.asarray(self.weights, dtype=float)\n'
              '\n'
              '\n'
              'def fit_nonnegative_ensemble_weights(oof_matrix, y):\n'
              '    matrix = np.asarray(oof_matrix, dtype=float)\n'
              '    target = np.asarray(y, dtype=float)\n'
              '    count = matrix.shape[1]\n'
              '    result = minimize(\n'
              '        lambda weights: np.mean((matrix @ weights - target) ** 2),\n'
              '        np.repeat(1.0 / count, count),\n'
              '        bounds=[(0.0, 1.0)] * count,\n'
              '        constraints={"type": "eq", "fun": lambda weights: weights.sum() - 1.0},\n'
              '        method="SLSQP",\n'
              '    )\n'
              '    if not result.success:\n'
              '        return np.repeat(1.0 / count, count)\n'
              '    weights = np.maximum(result.x, 0)\n'
              '    return weights / weights.sum()\n'
              '\n'
              '\n'
              'def build_base_models(seed: int = 42):\n'
              '    ridge = _numeric_pipeline(Ridge(alpha=10.0))\n'
              '    huber = _numeric_pipeline(HuberRegressor(epsilon=1.5, max_iter=3000), scale="robust")\n'
              '    lgbm = _numeric_pipeline(\n'
              '        LGBMRegressor(\n'
              '            n_estimators=350, learning_rate=0.03, num_leaves=7, max_depth=3,\n'
              '            min_child_samples=25, max_bin=31, reg_alpha=1.0, reg_lambda=10.0,\n'
              '            subsample=0.85, colsample_bytree=0.85, random_state=seed,\n'
              '            n_jobs=-1, verbose=-1,\n'
              '        ),\n'
              '        scale=None,\n'
              '    )\n'
              '    models = {\n'
              '        "Ridge": ridge,\n'
              '        "ElasticNet": _numeric_pipeline(ElasticNet(alpha=0.05, l1_ratio=0.25, max_iter=10000)),\n'
              '        "Huber": huber,\n'
              '        "GAM": Pipeline([\n'
              '            ("features", FurnaceFeatureEngineer()),\n'
              '            ("imputer", _imputer()),\n'
              '            ("splines", SplineTransformer(n_knots=4, degree=2, include_bias=False)),\n'
              '            ("scaler", StandardScaler()),\n'
              '            ("model", Ridge(alpha=10.0)),\n'
              '        ]),\n'
              '        "GPR": _numeric_pipeline(\n'
              '            GaussianProcessRegressor(\n'
              '                kernel=ConstantKernel(1.0) * Matern(length_scale=1.0, nu=1.5) + '
              'WhiteKernel(noise_level=0.2),\n'
              '                alpha=1e-6, normalize_y=True, n_restarts_optimizer=0, random_state=seed,\n'
              '            )\n'
              '        ),\n'
              '        "CatBoost": _numeric_pipeline(\n'
              '            CatBoostRegressor(\n'
              '                iterations=300, depth=4, learning_rate=0.03, loss_function="RMSE",\n'
              '                l2_leaf_reg=8.0, random_seed=seed, verbose=False, allow_writing_files=False,\n'
              '            ),\n'
              '            scale=None,\n'
              '        ),\n'
              '        "LightGBM": lgbm,\n'
              '    }\n'
              '    models["Ridge+LGBMResidual"] = ResidualBoostRegressor(ridge, lgbm)\n'
              '    models["Huber+LGBMResidual"] = ResidualBoostRegressor(huber, lgbm)\n'
              '    return models\n'
              '\n'
              '\n'
              'def build_tree_model_variants(seed: int = 42):\n'
              '    base = build_base_models(seed)\n'
              '    lgbm_variants = {}\n'
              '    for label, params in {\n'
              '        "very_small": {"num_leaves": 4, "max_depth": 2, "min_child_samples": 30, "reg_lambda": 15.0},\n'
              '        "small": {"num_leaves": 7, "max_depth": 3, "min_child_samples": 25, "reg_lambda": 10.0},\n'
              '        "medium_small": {"num_leaves": 10, "max_depth": 4, "min_child_samples": 20, "reg_lambda": '
              '8.0},\n'
              '    }.items():\n'
              '        lgbm_variants[label] = clone(base["LightGBM"]).set_params(\n'
              '            **{f"model__{key}": value for key, value in params.items()}\n'
              '        )\n'
              '    cat_variants = {}\n'
              '    for label, params in {\n'
              '        "depth3": {"depth": 3, "l2_leaf_reg": 10.0},\n'
              '        "depth4": {"depth": 4, "l2_leaf_reg": 8.0},\n'
              '    }.items():\n'
              '        cat_variants[label] = clone(base["CatBoost"]).set_params(\n'
              '            **{f"model__{key}": value for key, value in params.items()}\n'
              '        )\n'
              '    return {"LightGBM": lgbm_variants, "CatBoost": cat_variants}\n',
 'optimization.py': '"""Budget-matched random, genetic, and GPR expected-improvement search."""\n'
                    '\n'
                    'import math\n'
                    '\n'
                    'import numpy as np\n'
                    'import pandas as pd\n'
                    'from scipy.stats import norm\n'
                    'from sklearn.gaussian_process import GaussianProcessRegressor\n'
                    'from sklearn.gaussian_process.kernels import Matern, WhiteKernel\n'
                    '\n'
                    'from .data import DOOR_COUNT_COL, FEATURE_COLS, MELTING_COL, WEIGHT_COL\n'
                    'from .recommendation import CONTROLLABLE_COLS, score_candidates\n'
                    '\n'
                    '\n'
                    'def _sample(context, size: int, rng: np.random.Generator):\n'
                    '    frame = pd.DataFrame(index=range(size))\n'
                    '    frame[WEIGHT_COL] = context.total_weight\n'
                    '    frame[MELTING_COL] = context.melting_time\n'
                    '    for column, (low, high) in context.bounds.items():\n'
                    '        if column == DOOR_COUNT_COL:\n'
                    '            frame[column] = rng.integers(math.ceil(low), math.floor(high) + 1, size=size)\n'
                    '        else:\n'
                    '            frame[column] = rng.uniform(low, high, size=size)\n'
                    '    return frame[FEATURE_COLS]\n'
                    '\n'
                    '\n'
                    'def _set_attrs(result, name, budget, seed, context):\n'
                    '    result.attrs.update({\n'
                    '        "optimizer": name,\n'
                    '        "evaluations": len(result),\n'
                    '        "budget": int(budget),\n'
                    '        "seed": int(seed),\n'
                    '        "bounds": dict(context.bounds),\n'
                    '    })\n'
                    '    return result\n'
                    '\n'
                    '\n'
                    'def random_search(model, fold_models, context, feasibility, budget=600, seed=42):\n'
                    '    rng = np.random.default_rng(seed)\n'
                    '    result = score_candidates(model, fold_models, _sample(context, budget, rng), context, '
                    'feasibility)\n'
                    '    return _set_attrs(result, "random_search", budget, seed, context)\n'
                    '\n'
                    '\n'
                    'def genetic_search(model, fold_models, context, feasibility, budget=600, seed=42):\n'
                    '    rng = np.random.default_rng(seed)\n'
                    '    population_size = min(60, budget)\n'
                    '    population = _sample(context, population_size, rng)\n'
                    '    parts = []\n'
                    '    while sum(len(part) for part in parts) < budget:\n'
                    '        remaining = budget - sum(len(part) for part in parts)\n'
                    '        scored = score_candidates(model, fold_models, population.iloc[:remaining], context, '
                    'feasibility)\n'
                    '        parts.append(scored)\n'
                    '        if remaining <= population_size and remaining < len(population):\n'
                    '            break\n'
                    '        elite_count = max(4, population_size // 5)\n'
                    '        elites = scored.nsmallest(elite_count, '
                    '"penalized_objective")[CONTROLLABLE_COLS].reset_index(drop=True)\n'
                    '        children = [row.to_dict() for _, row in elites.iterrows()]\n'
                    '        while len(children) < population_size:\n'
                    '            a = elites.iloc[int(rng.integers(len(elites)))]\n'
                    '            b = elites.iloc[int(rng.integers(len(elites)))]\n'
                    '            child = {}\n'
                    '            for column, (low, high) in context.bounds.items():\n'
                    '                if column == DOOR_COUNT_COL:\n'
                    '                    value = a[column] if rng.random() < .5 else b[column]\n'
                    '                    if rng.random() < .20:\n'
                    '                        value += rng.choice([-1, 1])\n'
                    '                    child[column] = int(np.clip(round(value), math.ceil(low), math.floor(high)))\n'
                    '                else:\n'
                    '                    alpha = rng.random()\n'
                    '                    value = alpha * a[column] + (1 - alpha) * b[column]\n'
                    '                    if rng.random() < .20:\n'
                    '                        value += rng.normal(0, .08 * (high - low))\n'
                    '                    child[column] = float(np.clip(value, low, high))\n'
                    '            children.append(child)\n'
                    '        population = pd.DataFrame(children)\n'
                    '        population[WEIGHT_COL] = context.total_weight\n'
                    '        population[MELTING_COL] = context.melting_time\n'
                    '        population = population[FEATURE_COLS]\n'
                    '    result = pd.concat(parts, ignore_index=True).iloc[:budget].copy()\n'
                    '    return _set_attrs(result, "genetic_algorithm", budget, seed, context)\n'
                    '\n'
                    '\n'
                    'def _normalize_controls(frame, context):\n'
                    '    values = []\n'
                    '    for column in CONTROLLABLE_COLS:\n'
                    '        low, high = context.bounds[column]\n'
                    '        values.append((frame[column].to_numpy(dtype=float) - low) / max(high - low, 1e-12))\n'
                    '    return np.column_stack(values)\n'
                    '\n'
                    '\n'
                    'def bayesian_search(model, fold_models, context, feasibility, budget=600, seed=42):\n'
                    '    rng = np.random.default_rng(seed)\n'
                    '    initial_size = min(40, budget)\n'
                    '    evaluated = score_candidates(model, fold_models, _sample(context, initial_size, rng), '
                    'context, feasibility)\n'
                    '    while len(evaluated) < budget:\n'
                    '        remaining = budget - len(evaluated)\n'
                    '        batch_size = min(20, remaining)\n'
                    '        train = evaluated.nsmallest(min(250, len(evaluated)), "penalized_objective")\n'
                    '        X_train = _normalize_controls(train, context)\n'
                    '        y_train = train["penalized_objective"].to_numpy(dtype=float)\n'
                    '        y_mean, y_std = y_train.mean(), max(y_train.std(), 1e-9)\n'
                    '        gp = GaussianProcessRegressor(\n'
                    '            kernel=Matern(length_scale=np.ones(4), nu=1.5) + WhiteKernel(noise_level=.05),\n'
                    '            alpha=1e-6,\n'
                    '            normalize_y=False,\n'
                    '            optimizer=None,\n'
                    '            random_state=seed,\n'
                    '        ).fit(X_train, (y_train - y_mean) / y_std)\n'
                    '        pool = _sample(context, max(500, batch_size * 20), rng)\n'
                    '        mean, std = gp.predict(_normalize_controls(pool, context), return_std=True)\n'
                    '        best = float(((y_train - y_mean) / y_std).min())\n'
                    '        improvement = best - mean\n'
                    '        z = improvement / np.maximum(std, 1e-12)\n'
                    '        expected_improvement = improvement * norm.cdf(z) + std * norm.pdf(z)\n'
                    '        selected = pool.iloc[np.argsort(expected_improvement)[-batch_size:]]\n'
                    '        scored = score_candidates(model, fold_models, selected, context, feasibility)\n'
                    '        evaluated = pd.concat([evaluated, scored], ignore_index=True)\n'
                    '    result = evaluated.iloc[:budget].copy()\n'
                    '    return _set_attrs(result, "bayesian_optimization", budget, seed, context)\n'
                    '\n'
                    '\n'
                    'def compare_optimizers(model, fold_models, context, feasibility, common_budget=600, seeds=(0, 1, '
                    '2), keep_results=True):\n'
                    '    functions = (\n'
                    '        ("random_search", random_search),\n'
                    '        ("genetic_algorithm", genetic_search),\n'
                    '        ("bayesian_optimization", bayesian_search),\n'
                    '    )\n'
                    '    rows = []\n'
                    '    results = {}\n'
                    '    for seed in seeds:\n'
                    '        for name, function in functions:\n'
                    '            result = function(model, fold_models, context, feasibility, budget=common_budget, '
                    'seed=int(seed))\n'
                    '            if keep_results:\n'
                    '                results[(name, int(seed))] = result\n'
                    '            best = result.nsmallest(1, "penalized_objective").iloc[0]\n'
                    '            rows.append({\n'
                    '                "optimizer": name,\n'
                    '                "seed": int(seed),\n'
                    '                "evaluations": len(result),\n'
                    '                "best_objective": float(best["penalized_objective"]),\n'
                    '                "safe_candidate_rate": float((\n'
                    '                    (result["estimated_saving_vs_actual_baseline"] > 0)\n'
                    '                    & (result["fold_consensus_rate"] >= 2/3)\n'
                    '                    & result["historically_feasible"]\n'
                    '                    & ~result["boundary_hit"]\n'
                    '                ).mean()),\n'
                    '                **{f"best__{column}": float(best[column]) for column in CONTROLLABLE_COLS},\n'
                    '            })\n'
                    '    runs = pd.DataFrame(rows)\n'
                    '    selected = str(runs.groupby("optimizer")["best_objective"].median().idxmin())\n'
                    '    response = {"runs": runs, "selected_optimizer": selected}\n'
                    '    if keep_results:\n'
                    '        response["results"] = results\n'
                    '    return response\n',
 'recommendation.py': '"""Historically grounded recommendation scoring and explicit safety gates."""\n'
                      '\n'
                      'from dataclasses import dataclass\n'
                      'import math\n'
                      '\n'
                      'import numpy as np\n'
                      'import pandas as pd\n'
                      'from sklearn.impute import SimpleImputer\n'
                      'from sklearn.neighbors import NearestNeighbors\n'
                      'from sklearn.preprocessing import RobustScaler\n'
                      '\n'
                      'from .data import (\n'
                      '    DOOR_COUNT_COL,\n'
                      '    DOOR_DURATION_COL,\n'
                      '    FEATURE_COLS,\n'
                      '    MELTING_COL,\n'
                      '    SOLID_COL,\n'
                      '    TARGET_COL,\n'
                      '    WAIT_COL,\n'
                      '    WEIGHT_COL,\n'
                      ')\n'
                      '\n'
                      '\n'
                      'CONTROLLABLE_COLS = [SOLID_COL, WAIT_COL, DOOR_COUNT_COL, DOOR_DURATION_COL]\n'
                      '\n'
                      '\n'
                      '@dataclass(frozen=True)\n'
                      'class RecommendationContext:\n'
                      '    reference_df: pd.DataFrame\n'
                      '    total_weight: float\n'
                      '    melting_time: float\n'
                      '    bounds: dict[str, tuple[float, float]]\n'
                      '    baseline_controls: dict[str, float]\n'
                      '    actual_baseline_gas: float\n'
                      '    historical_low_gas_median: float\n'
                      '    target_iqr: float\n'
                      '    trust_ratio: float\n'
                      '    weight_tolerance: float | None\n'
                      '    low_confidence: bool\n'
                      '\n'
                      '\n'
                      '@dataclass(frozen=True)\n'
                      'class SafetyDecision:\n'
                      '    passes: bool\n'
                      '    grade: str\n'
                      '    reasons: tuple[str, ...]\n'
                      '\n'
                      '\n'
                      'def build_recommendation_context(df, total_weight: float, trust_ratio: float = 0.10):\n'
                      '    reference = pd.DataFrame()\n'
                      '    tolerance_used = None\n'
                      '    for tolerance in (0.05, 0.10):\n'
                      '        reference = df[df[WEIGHT_COL].between(total_weight * (1 - tolerance), total_weight * (1 '
                      '+ tolerance))]\n'
                      '        if len(reference) >= 20:\n'
                      '            tolerance_used = tolerance\n'
                      '            break\n'
                      '    low_confidence = len(reference) < 20\n'
                      '    if low_confidence:\n'
                      '        reference = df.copy()\n'
                      '    low_count = max(1, math.ceil(len(reference) * 0.20))\n'
                      '    low_gas = reference.nsmallest(low_count, TARGET_COL)\n'
                      '    baseline = {column: float(low_gas[column].median()) for column in CONTROLLABLE_COLS}\n'
                      '    bounds = {}\n'
                      '    for column in CONTROLLABLE_COLS:\n'
                      '        history_low = float(reference[column].quantile(0.05))\n'
                      '        history_high = float(reference[column].quantile(0.95))\n'
                      '        center = baseline[column]\n'
                      '        trust_low = center * (1 - trust_ratio)\n'
                      '        trust_high = center * (1 + trust_ratio)\n'
                      '        low, high = max(history_low, trust_low), min(history_high, trust_high)\n'
                      '        if column == DOOR_COUNT_COL:\n'
                      '            low, high = float(math.ceil(low)), float(math.floor(high))\n'
                      '        if low >= high:\n'
                      '            raise ValueError(f"参数 {column} 的历史范围与信任区域没有有效交集。")\n'
                      '        bounds[column] = (low, high)\n'
                      '    return RecommendationContext(\n'
                      '        reference_df=reference.copy(),\n'
                      '        total_weight=float(total_weight),\n'
                      '        melting_time=float(reference[MELTING_COL].median()),\n'
                      '        bounds=bounds,\n'
                      '        baseline_controls=baseline,\n'
                      '        actual_baseline_gas=float(reference[TARGET_COL].median()),\n'
                      '        historical_low_gas_median=float(low_gas[TARGET_COL].median()),\n'
                      '        target_iqr=float(df[TARGET_COL].quantile(.75) - df[TARGET_COL].quantile(.25)),\n'
                      '        trust_ratio=float(trust_ratio),\n'
                      '        weight_tolerance=tolerance_used,\n'
                      '        low_confidence=low_confidence,\n'
                      '    )\n'
                      '\n'
                      '\n'
                      'def fit_feasibility_reference(df):\n'
                      '    imputer = SimpleImputer(strategy="median")\n'
                      '    scaler = RobustScaler()\n'
                      '    transformed = scaler.fit_transform(imputer.fit_transform(df[FEATURE_COLS]))\n'
                      '    neighbors = min(6, len(df))\n'
                      '    model = NearestNeighbors(n_neighbors=neighbors).fit(transformed)\n'
                      '    distance = model.kneighbors(transformed, return_distance=True)[0]\n'
                      '    reference_index = min(5, distance.shape[1] - 1)\n'
                      '    threshold = max(float(np.quantile(distance[:, reference_index], .95)), 1e-12)\n'
                      '    return {\n'
                      '        "imputer": imputer,\n'
                      '        "scaler": scaler,\n'
                      '        "neighbors": model,\n'
                      '        "distance_threshold": threshold,\n'
                      '        "reference_rows": len(df),\n'
                      '    }\n'
                      '\n'
                      '\n'
                      'def score_candidates(candidate_model, fold_models, candidates, context, feasibility):\n'
                      '    frame = candidates[FEATURE_COLS].apply(pd.to_numeric, '
                      'errors="coerce").reset_index(drop=True)\n'
                      '    if frame.isna().any().any():\n'
                      '        raise ValueError("候选参数必须是完整数值。")\n'
                      '    result = frame.copy()\n'
                      '    result["full_model_prediction"] = np.asarray(candidate_model.predict(frame), dtype=float)\n'
                      '    fold_matrix = np.vstack([model.predict(frame) for model in fold_models])\n'
                      '    result["fold_prediction_median"] = np.median(fold_matrix, axis=0)\n'
                      '    result["prediction_std"] = fold_matrix.std(axis=0)\n'
                      '    result["fold_consensus_rate"] = (fold_matrix < context.actual_baseline_gas).mean(axis=0)\n'
                      '    result["conservative_predicted_gas"] = result["fold_prediction_median"] + '
                      'result["prediction_std"]\n'
                      '    result["estimated_saving_vs_actual_baseline"] = context.actual_baseline_gas - '
                      'result["conservative_predicted_gas"]\n'
                      '\n'
                      '    scaled = feasibility["scaler"].transform(feasibility["imputer"].transform(frame))\n'
                      '    neighbor_count = min(5, feasibility["reference_rows"])\n'
                      '    distance = feasibility["neighbors"].kneighbors(\n'
                      '        scaled, n_neighbors=neighbor_count, return_distance=True\n'
                      '    )[0][:, -1]\n'
                      '    result["normalized_knn_distance"] = distance / feasibility["distance_threshold"]\n'
                      '    result["historically_feasible"] = result["normalized_knn_distance"] <= 1.0\n'
                      '\n'
                      '    boundary_count = np.zeros(len(result), dtype=int)\n'
                      '    change_penalty = np.zeros(len(result), dtype=float)\n'
                      '    for column, (low, high) in context.bounds.items():\n'
                      '        width = max(high - low, 1e-12)\n'
                      '        hit = ((result[column] - low) <= .02 * width) | ((high - result[column]) <= .02 * '
                      'width)\n'
                      '        result[f"boundary__{column}"] = hit\n'
                      '        boundary_count += hit.astype(int)\n'
                      '        baseline = context.baseline_controls[column]\n'
                      '        change_penalty += np.abs(result[column] - baseline) / max(abs(baseline), 1.0)\n'
                      '    result["boundary_hit_count"] = boundary_count\n'
                      '    result["boundary_hit"] = boundary_count > 0\n'
                      '    result["mean_relative_change"] = change_penalty / len(CONTROLLABLE_COLS)\n'
                      '    result["within_trust_region"] = True\n'
                      '    result["penalized_objective"] = (\n'
                      '        result["conservative_predicted_gas"]\n'
                      '        + context.target_iqr * np.maximum(0, result["normalized_knn_distance"] - 1)\n'
                      '        + context.target_iqr * .35 * result["boundary_hit_count"]\n'
                      '        + context.target_iqr * .10 * result["mean_relative_change"]\n'
                      '    )\n'
                      '    return result\n'
                      '\n'
                      '\n'
                      'def safety_gate(row, context: RecommendationContext) -> SafetyDecision:\n'
                      '    reasons = []\n'
                      '    if float(row["conservative_predicted_gas"]) >= context.actual_baseline_gas:\n'
                      '        reasons.append("保守预测未低于相似历史实际气耗基准")\n'
                      '    if float(row["fold_consensus_rate"]) < (2 / 3):\n'
                      '        reasons.append("少于2/3时间折认为可以节气")\n'
                      '    if not bool(row["historically_feasible"]):\n'
                      '        reasons.append("历史近邻支持不足")\n'
                      '    if bool(row["boundary_hit"]):\n'
                      '        reasons.append("推荐命中搜索边界")\n'
                      '    if not bool(row.get("within_trust_region", True)):\n'
                      '        reasons.append("推荐超出信任区域")\n'
                      '    passes = not reasons\n'
                      '    return SafetyDecision(passes=passes, grade="A" if passes else "C", reasons=tuple(reasons))\n'
                      '\n'
                      '\n'
                      'def historical_fallback(context: RecommendationContext):\n'
                      '    return {\n'
                      '        "source": "historical_similar_low_gas",\n'
                      '        "recommendation": dict(context.baseline_controls),\n'
                      '        "actual_gas_median": context.historical_low_gas_median,\n'
                      '        "actual_baseline_gas": context.actual_baseline_gas,\n'
                      '        "safety_pass": False,\n'
                      '        "safety_grade": "fallback",\n'
                      '    }\n'
                      '\n'
                      '\n'
                      'def recommend_or_fallback(scored_candidates, context: RecommendationContext):\n'
                      '    ordered = scored_candidates.sort_values("penalized_objective")\n'
                      '    for _, row in ordered.iterrows():\n'
                      '        decision = safety_gate(row, context)\n'
                      '        if decision.passes:\n'
                      '            return {\n'
                      '                "source": "model_optimization",\n'
                      '                "recommendation": {column: float(row[column]) for column in '
                      'CONTROLLABLE_COLS},\n'
                      '                "conservative_predicted_gas": float(row["conservative_predicted_gas"]),\n'
                      '                "estimated_saving_vs_actual_baseline": '
                      'float(row["estimated_saving_vs_actual_baseline"]),\n'
                      '                "actual_baseline_gas": context.actual_baseline_gas,\n'
                      '                "historical_low_gas_median": context.historical_low_gas_median,\n'
                      '                "fold_consensus_rate": float(row["fold_consensus_rate"]),\n'
                      '                "prediction_std": float(row["prediction_std"]),\n'
                      '                "normalized_knn_distance": float(row["normalized_knn_distance"]),\n'
                      '                "safety_pass": True,\n'
                      '                "safety_grade": decision.grade,\n'
                      '                "safety_reasons": [],\n'
                      '            }\n'
                      '    fallback = historical_fallback(context)\n'
                      '    fallback["safety_reasons"] = ["没有模型候选同时通过全部离线安全门"]\n'
                      '    return fallback\n',
 'uncertainty.py': '"""Conformal intervals and chronological fold disagreement."""\n'
                   '\n'
                   'import numpy as np\n'
                   '\n'
                   '\n'
                   'def conformal_radius(y_true, oof_prediction, coverage: float = 0.90) -> float:\n'
                   '    y = np.asarray(y_true, dtype=float)\n'
                   '    prediction = np.asarray(oof_prediction, dtype=float)\n'
                   '    valid = np.isfinite(prediction)\n'
                   '    if not valid.any():\n'
                   '        raise ValueError("没有可用于 conformal 校准的 OOF 预测。")\n'
                   '    return float(np.quantile(np.abs(y[valid] - prediction[valid]), coverage))\n'
                   '\n'
                   '\n'
                   'def prediction_interval(prediction, radius: float):\n'
                   '    values = np.asarray(prediction, dtype=float)\n'
                   '    return np.maximum(0.0, values - float(radius)), values + float(radius)\n'
                   '\n'
                   '\n'
                   'def fold_prediction_summary(fold_models, X):\n'
                   '    matrix = np.vstack([model.predict(X) for model in fold_models])\n'
                   '    return {\n'
                   '        "matrix": matrix,\n'
                   '        "median": np.median(matrix, axis=0),\n'
                   '        "mean": matrix.mean(axis=0),\n'
                   '        "std": matrix.std(axis=0),\n'
                   '    }\n',
 'validation.py': '"""Large-window chronological validation and locked audit helpers."""\n'
                  '\n'
                  'from dataclasses import dataclass\n'
                  '\n'
                  'import numpy as np\n'
                  'from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\n'
                  '\n'
                  '\n'
                  'def large_window_splits(n_dev: int = 248):\n'
                  '    if n_dev != 248:\n'
                  '        raise ValueError("当前正式设计要求开发区恰好为248炉。")\n'
                  '    boundaries = [(148, 181), (181, 214), (214, 248)]\n'
                  '    return [(np.arange(train_end), np.arange(train_end, test_end)) for train_end, test_end in '
                  'boundaries]\n'
                  '\n'
                  '\n'
                  'def regression_metrics(y_true, y_pred) -> dict:\n'
                  '    y = np.asarray(y_true, dtype=float)\n'
                  '    p = np.asarray(y_pred, dtype=float)\n'
                  '    absolute = np.abs(y - p)\n'
                  '    denominator = np.maximum(np.abs(y), 1e-12)\n'
                  '    return {\n'
                  '        "mae": float(mean_absolute_error(y, p)),\n'
                  '        "rmse": float(mean_squared_error(y, p) ** 0.5),\n'
                  '        "wape": float(absolute.sum() / denominator.sum()),\n'
                  '        "r2": float(r2_score(y, p)),\n'
                  '        "error_gt_10pct_rate": float((absolute / denominator > 0.10).mean()),\n'
                  '    }\n'
                  '\n'
                  '\n'
                  'def bootstrap_metric_interval(y_true, y_pred, metric: str, seed: int, n_boot: int = 1000):\n'
                  '    y = np.asarray(y_true, dtype=float)\n'
                  '    p = np.asarray(y_pred, dtype=float)\n'
                  '    rng = np.random.default_rng(seed)\n'
                  '    values = []\n'
                  '    for _ in range(n_boot):\n'
                  '        index = rng.integers(0, len(y), len(y))\n'
                  '        values.append(regression_metrics(y[index], p[index])[metric])\n'
                  '    return float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))\n'
                  '\n'
                  '\n'
                  '@dataclass\n'
                  'class LockedAudit:\n'
                  '    _values: object\n'
                  '    _frozen: bool = False\n'
                  '\n'
                  '    def freeze(self):\n'
                  '        self._frozen = True\n'
                  '\n'
                  '    def values(self):\n'
                  '        if not self._frozen:\n'
                  '            raise RuntimeError("Locked audit cannot be read before freeze().")\n'
                  '        return self._values\n'}

import importlib
import sys
RUNTIME_ROOT = OUTPUT_ROOT / 'runtime'
PACKAGE_DIR = RUNTIME_ROOT / 'furnace_complete_runtime'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
for filename, source_text in RUNTIME_SOURCES.items():
    (PACKAGE_DIR / filename).write_text(source_text, encoding='utf-8')
if str(RUNTIME_ROOT) not in sys.path:
    sys.path.insert(0, str(RUNTIME_ROOT))
importlib.invalidate_caches()
print(f'已从 notebook 内嵌快照生成独立运行时: {PACKAGE_DIR}')

## Excel 数据质量检查

In [ ]:
from furnace_complete_runtime.data import (
    FEATURE_COLS, TARGET_COL, chronological_dev_lock_split, load_batch_data, mark_target_outliers,
)
from furnace_complete_runtime.validation import large_window_splits
df = mark_target_outliers(load_batch_data(EXCEL_PATH))
dev, locked = chronological_dev_lock_split(df, lock_size=44)
splits = large_window_splits(len(dev))
assert len(df) == 292
assert len(dev) == 248
assert len(locked) == 44
expected_windows = [(0, 148, 148, 181), (0, 181, 181, 214), (0, 214, 214, 248)]
observed_windows = [(int(tr.min()), int(tr.max()) + 1, int(va.min()), int(va.max()) + 1) for tr, va in splits]
assert observed_windows == expected_windows
display(df.head())
display({'总炉次': len(df), '开发炉次': len(dev), '锁定炉次': len(locked), '高气耗异常标记数（保留）': int(df['is_high_gas_outlier'].sum())})

## 三个时间折与锁定集

In [ ]:
import pandas as pd
fold_design = pd.DataFrame([
    {'fold': i + 1, 'train_start': w[0], 'train_end_exclusive': w[1], 'valid_start': w[2], 'valid_end_exclusive': w[3], 'train_size': w[1]-w[0], 'valid_size': w[3]-w[2]}
    for i, w in enumerate(observed_windows)
])
display(fold_design)
print('已验证：每个验证窗口都严格晚于训练窗口，没有未来数据泄漏。')

## 完整模型矩阵与开发阶段选模

In [ ]:
from furnace_complete_runtime.experiment import random_cv_reference, run_model_matrix
print('开始从 Excel 训练完整模型矩阵；锁定44炉此时不参与选模。')
experiment = run_model_matrix(dev, splits)
selected_before_lock = experiment.selected_name
model_cv_summary = experiment.summary.copy()
model_cv_summary['selected_before_lock'] = model_cv_summary['candidate'].eq(selected_before_lock)
chronological_fold_metrics = pd.concat(
    [result.fold_metrics for result in experiment.results.values()], ignore_index=True
)
tree_tuning_summary = experiment.tree_tuning_summary.copy()
random_cv_reference_table = random_cv_reference(experiment, n_splits=3, seed=42)
required_candidates = {'OOFEnsemble', 'Ridge+LGBMResidual__direct', 'Huber+LGBMResidual__direct', 'GPR__direct', 'LightGBM__direct'}
assert required_candidates.issubset(set(model_cv_summary['candidate']))
assert model_cv_summary['candidate'].nunique() == 19
assert model_cv_summary.loc[model_cv_summary['selected_before_lock'], 'candidate'].item() == selected_before_lock
experiment.freeze()
display(model_cv_summary.round(4))
display(tree_tuning_summary.round(4))
display(random_cv_reference_table.head(12).round(4))
print(f'开发阶段 Champion: {selected_before_lock}')
print('包含 conformal_radius_90、时间折波动；GPR 行另含 native_model_std_mean。')

## 最后44炉锁定审计

开发阶段 Champion 已经在不查看最后44炉结果的情况下冻结。下面只做一次审计。锁定集不能反向参与选模；否则它就不再是独立测试集。

In [ ]:
from furnace_complete_runtime.experiment import audit_frozen_models
locked_audit = audit_frozen_models(experiment, locked)
assert len(locked_audit) == len(model_cv_summary) == 19
assert int(len(locked)) == 44
locked_rmse_winner = str(locked_audit.sort_values('rmse').iloc[0]['candidate'])
interpretation = pd.DataFrame([
    {'角色': '开发阶段 Champion', '模型': selected_before_lock, '用途': '按三个时间折预先选出的预测模型'},
    {'角色': '锁定集 RMSE 优胜者', '模型': locked_rmse_winner, '用途': '最后44炉上的审计优胜者，不反向改写选模'},
])
required_audit_columns = {
    'mae', 'rmse', 'wape', 'r2', 'error_gt_10pct_rate',
    'rmse_bootstrap_low95', 'rmse_bootstrap_high95', 'interval_coverage_90',
}
assert required_audit_columns.issubset(locked_audit.columns)
display(locked_audit.sort_values('rmse').round(4))
display(interpretation)
print(f'开发阶段 Champion: {selected_before_lock}')
print(f'锁定集 RMSE 优胜者: {locked_rmse_winner}')
print('锁定集不能反向参与选模。')

## 86000 kg 历史基准推导

In [ ]:
import math
import warnings
import numpy as np
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from furnace_complete_runtime.data import (
    DOOR_COUNT_COL, MELTING_COL, WEIGHT_COL,
)
from furnace_complete_runtime.recommendation import (
    CONTROLLABLE_COLS, build_recommendation_context, fit_feasibility_reference,
    historical_fallback, recommend_or_fallback, safety_gate,
)
TARGET_WEIGHT = 86000.0
recommendation_context = build_recommendation_context(df, TARGET_WEIGHT, trust_ratio=0.10)
similar_history_rows = len(recommendation_context.reference_df)
low_gas_20pct_rows = max(1, math.ceil(similar_history_rows * 0.20))
actual_baseline_gas = recommendation_context.actual_baseline_gas
baseline_table = pd.DataFrame([
    {'项目': '相似历史炉次数', '数值': similar_history_rows},
    {'项目': '重量容差', '数值': recommendation_context.weight_tolerance},
    {'项目': '低气耗20%炉次数', '数值': low_gas_20pct_rows},
    {'项目': '相似历史实际气耗中位数', '数值': actual_baseline_gas},
    {'项目': '低气耗20%实际气耗中位数', '数值': recommendation_context.historical_low_gas_median},
])
control_baseline_table = pd.DataFrame([
    {'参数': column, '历史低气耗中位数': value}
    for column, value in recommendation_context.baseline_controls.items()
])
assert similar_history_rows == 97
assert low_gas_20pct_rows == 20
assert abs(actual_baseline_gas - 3701.28) < 1e-9
display(baseline_table)
display(control_baseline_table.round(4))
print('历史低气耗参数分别取中位数，不保证来自同一炉；后续用kNN检查联合组合。')
feasibility_reference = fit_feasibility_reference(df)
recommendation_candidate_names = [
    'LightGBM__direct', 'GPR__direct', 'Ridge+LGBMResidual__direct',
    'Huber+LGBMResidual__direct', 'OOFEnsemble',
]
full_recommendation_models = {}
for candidate_name in recommendation_candidate_names:
    model = clone(experiment.results[candidate_name].estimator_template)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        model.fit(df[FEATURE_COLS], df[TARGET_COL])
    full_recommendation_models[candidate_name] = model
print('已在锁定审计完成后，用全部292炉拟合最终推荐候选模型。')

## 宽范围搜索诊断

该部分使用 historical 5%-95% 宽范围，只用于解释为什么优化器可能寻找边界或高不确定性参数，不把结果直接作为生产推荐。

In [ ]:
from dataclasses import replace
from furnace_complete_runtime.optimization import genetic_search
broad_bounds = {}
for column in CONTROLLABLE_COLS:
    low = float(recommendation_context.reference_df[column].quantile(0.05))
    high = float(recommendation_context.reference_df[column].quantile(0.95))
    if column == DOOR_COUNT_COL:
        low, high = float(math.ceil(low)), float(math.floor(high))
    broad_bounds[column] = (low, high)
broad_context = replace(recommendation_context, bounds=broad_bounds, trust_ratio=1.0)
broad_scored = genetic_search(
    full_recommendation_models['LightGBM__direct'],
    experiment.results['LightGBM__direct'].fold_models,
    broad_context, feasibility_reference, budget=600, seed=42,
)
broad_best = broad_scored.sort_values('penalized_objective').iloc[0]
broad_search_diagnostics = pd.DataFrame([{
    'candidate': 'LightGBM__direct',
    'search_range': 'historical 5%-95%',
    'conservative_predicted_gas': float(broad_best['conservative_predicted_gas']),
    'prediction_std': float(broad_best['prediction_std']),
    'normalized_knn_distance': float(broad_best['normalized_knn_distance']),
    'boundary_hit': bool(broad_best['boundary_hit']),
    'boundary_pass': not bool(broad_best['boundary_hit']),
    'uncertainty_pass_old_rule': bool(broad_best['prediction_std'] <= 0.50 * recommendation_context.target_iqr),
    **{column: float(broad_best[column]) for column in CONTROLLABLE_COLS},
}])
display(broad_search_diagnostics.round(4))
print('宽范围结果是风险诊断；低预测值若命中边界或折间分歧大，不进入生产候选。')

## 生产候选推荐与安全门

In [ ]:
from furnace_complete_runtime.optimization import (
    bayesian_search, compare_optimizers, genetic_search, random_search,
)
SEARCH_BUDGET = 600
DEPLOYMENT_SEED = 42
OPTIMIZER_FUNCTIONS = {
    'random_search': random_search,
    'genetic_algorithm': genetic_search,
    'bayesian_optimization': bayesian_search,
}
optimizer_parts = []
recommendation_rows = []
recommendation_payloads = {}
deployment_scored = {}
for candidate_name in recommendation_candidate_names:
    fold_models = experiment.results[candidate_name].fold_models
    comparison = compare_optimizers(
        full_recommendation_models[candidate_name], fold_models,
        recommendation_context, feasibility_reference,
        common_budget=SEARCH_BUDGET, seeds=(0, 1, 2), keep_results=False,
    )
    runs = comparison['runs'].copy()
    runs['candidate'] = candidate_name
    optimizer_parts.append(runs)
    selected_optimizer = comparison['selected_optimizer']
    scored = OPTIMIZER_FUNCTIONS[selected_optimizer](
        full_recommendation_models[candidate_name], fold_models,
        recommendation_context, feasibility_reference,
        budget=SEARCH_BUDGET, seed=DEPLOYMENT_SEED,
    )
    deployment_scored[candidate_name] = scored
    payload = recommend_or_fallback(scored, recommendation_context)
    payload['candidate'] = candidate_name
    payload['optimizer'] = selected_optimizer
    recommendation_payloads[candidate_name] = payload
    ordered = scored.sort_values('penalized_objective')
    safe_row = next((row for _, row in ordered.iterrows() if safety_gate(row, recommendation_context).passes), None)
    if safe_row is None:
        fallback = historical_fallback(recommendation_context)
        recommendation_rows.append({
            'candidate': candidate_name, 'optimizer': selected_optimizer, 'source': fallback['source'],
            'savings_pass': False, 'consensus_pass': False, 'feasibility_pass': False,
            'boundary_pass': False, 'trust_region_pass': False, 'safety_pass': False,
            'safety_reasons': '没有候选同时通过全部安全门',
            **fallback['recommendation'],
        })
        continue
    savings_pass = bool(safe_row['conservative_predicted_gas'] < actual_baseline_gas)
    consensus_pass = bool(safe_row['fold_consensus_rate'] >= 2 / 3)
    feasibility_pass = bool(safe_row['historically_feasible'])
    boundary_pass = not bool(safe_row['boundary_hit'])
    trust_region_pass = bool(safe_row['within_trust_region'])
    safety_pass = all([savings_pass, consensus_pass, feasibility_pass, boundary_pass, trust_region_pass])
    recommendation_rows.append({
        'candidate': candidate_name, 'optimizer': selected_optimizer, 'source': 'model_optimization',
        'actual_baseline_gas': actual_baseline_gas,
        'conservative_predicted_gas': float(safe_row['conservative_predicted_gas']),
        'estimated_saving_vs_actual_baseline': float(safe_row['estimated_saving_vs_actual_baseline']),
        'fold_consensus_rate': float(safe_row['fold_consensus_rate']),
        'prediction_std': float(safe_row['prediction_std']),
        'normalized_knn_distance': float(safe_row['normalized_knn_distance']),
        'savings_pass': savings_pass, 'consensus_pass': consensus_pass,
        'feasibility_pass': feasibility_pass, 'boundary_pass': boundary_pass,
        'trust_region_pass': trust_region_pass, 'safety_pass': safety_pass,
        'safety_reasons': '',
        **{column: float(safe_row[column]) for column in CONTROLLABLE_COLS},
    })
optimizer_comparison = pd.concat(optimizer_parts, ignore_index=True)
recommendation_summary = pd.DataFrame(recommendation_rows)
assert set(optimizer_comparison['evaluations']) == {SEARCH_BUDGET}
assert set(optimizer_comparison['seed']) == {0, 1, 2}
assert (recommendation_summary[DOOR_COUNT_COL].dropna() % 1 == 0).all()
passing = recommendation_summary[recommendation_summary['safety_pass']]
if passing.empty:
    production_recommendation = historical_fallback(recommendation_context)
    production_recommendation['candidate'] = selected_before_lock
    production_recommendation['optimizer'] = 'historical_fallback'
    production_recommendation_model = selected_before_lock
else:
    production_recommendation_model = str(
        passing.sort_values('estimated_saving_vs_actual_baseline', ascending=False).iloc[0]['candidate']
    )
    production_recommendation = recommendation_payloads[production_recommendation_model]
display(optimizer_comparison.round(4))
display(recommendation_summary.round(4))
display(pd.DataFrame([production_recommendation]).round(4))
print('安全门要求：节气、至少2/3时间折共识、历史可行、不命中边界、位于±10%信任区域。')
print('离线预计节省不等于工厂实际节省，必须经过现场受控试验。')

## joblib 保存与重载复现

In [ ]:
import json
from datetime import datetime, timezone
from furnace_complete_runtime.artifacts import save_bundle
selected_result = experiment.results[selected_before_lock]
prediction_model = clone(selected_result.estimator_template)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    prediction_model.fit(df[FEATURE_COLS], df[TARGET_COL])
bundle = {
    'artifact_version': 'advanced-furnace-1.0',
    'notebook_artifact_version': 'furnace-complete-notebook-1.0',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'feature_cols': list(FEATURE_COLS),
    'training_batches': len(df), 'dev_batches': len(dev), 'locked_batches': len(locked),
    'selected_model_name': selected_before_lock,
    'locked_rmse_winner': locked_rmse_winner,
    'prediction_model': prediction_model,
    'prediction_fold_models': selected_result.fold_models,
    'conformal_radius_90': selected_result.conformal_radius_90,
    'model_cv_summary': model_cv_summary,
    'tree_tuning_summary': tree_tuning_summary,
    'random_cv_reference': random_cv_reference_table,
    'locked_audit': locked_audit,
    'training_data': df,
    'recommendation_model_name': production_recommendation_model,
    'recommendation_model': full_recommendation_models[production_recommendation_model],
    'recommendation_fold_models': experiment.results[production_recommendation_model].fold_models,
    'recommendation_optimizer': production_recommendation['optimizer'],
    'recommendation_seed': DEPLOYMENT_SEED,
    'recommendation_trust_ratio': 0.10,
    'feasibility_reference': feasibility_reference,
    'production_recommendation_86000': production_recommendation,
    'offline_savings_disclaimer': '离线预计节省不等于工厂实际节省，必须经过现场受控试验。',
}
ARTIFACT_PATH = ARTIFACTS_DIR / 'furnace_complete_bundle.joblib'
save_bundle(bundle, ARTIFACT_PATH)
model_cv_summary.to_csv(REPORTS_DIR / 'model_cv_summary.csv', index=False)
chronological_fold_metrics.to_csv(REPORTS_DIR / 'chronological_fold_metrics.csv', index=False)
tree_tuning_summary.to_csv(REPORTS_DIR / 'tree_tuning_summary.csv', index=False)
random_cv_reference_table.to_csv(REPORTS_DIR / 'random_cv_reference.csv', index=False)
locked_audit.to_csv(REPORTS_DIR / 'locked_audit.csv', index=False)
broad_search_diagnostics.to_csv(REPORTS_DIR / 'broad_search_diagnostics.csv', index=False)
optimizer_comparison.to_csv(REPORTS_DIR / 'optimizer_comparison.csv', index=False)
recommendation_summary.to_csv(REPORTS_DIR / 'recommendation_summary.csv', index=False)
run_summary = {
    'notebook_artifact_version': bundle['notebook_artifact_version'],
    'training_batches': len(df), 'dev_batches': len(dev), 'locked_batches': len(locked),
    'selected_before_lock': selected_before_lock,
    'locked_rmse_winner': locked_rmse_winner,
    'recommendation_model': production_recommendation_model,
    'recommendation_optimizer': production_recommendation['optimizer'],
    'recommendation_seed': DEPLOYMENT_SEED,
    'production_recommendation_86000': production_recommendation,
    'artifact_path': str(ARTIFACT_PATH),
    'offline_savings_disclaimer': bundle['offline_savings_disclaimer'],
}
(REPORTS_DIR / 'run_summary.json').write_text(
    json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'已保存独立模型包: {ARTIFACT_PATH}')

In [ ]:
# 模拟新的推理进程：清除运行时模块，再从 notebook 生成的 runtime 目录重新导入。
sample_X = df[FEATURE_COLS].iloc[[0]].copy()
prediction_before = np.asarray(bundle['prediction_model'].predict(sample_X), dtype=float)
for module_name in list(sys.modules):
    if module_name == 'furnace_complete_runtime' or module_name.startswith('furnace_complete_runtime.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
reloaded_artifacts = importlib.import_module('furnace_complete_runtime.artifacts')
reloaded_bundle = reloaded_artifacts.load_bundle(ARTIFACT_PATH)
prediction_after = reloaded_artifacts.predict_bundle(reloaded_bundle, sample_X)['predicted_gas'].to_numpy()
np.testing.assert_allclose(prediction_before, prediction_after, rtol=0, atol=1e-10)
prediction_round_trip_atol_1e_10 = True
replayed_recommendation = reloaded_artifacts.recommend_bundle(
    reloaded_bundle, TARGET_WEIGHT, budget=SEARCH_BUDGET, seed=DEPLOYMENT_SEED
)
expected_recommendation = bundle['production_recommendation_86000']
for column in CONTROLLABLE_COLS:
    assert abs(replayed_recommendation['recommendation'][column] - expected_recommendation['recommendation'][column]) <= 1e-10
assert abs(replayed_recommendation['conservative_predicted_gas'] - expected_recommendation['conservative_predicted_gas']) <= 1e-10
recommendation_round_trip_atol_1e_10 = True
def rejected(frame):
    try:
        reloaded_artifacts.predict_bundle(reloaded_bundle, frame)
        return False
    except ValueError:
        return True
malformed_missing_rejected = rejected(sample_X.drop(columns=FEATURE_COLS[0]))
malformed_extra_rejected = rejected(sample_X.assign(额外字段=1))
nonnumeric = sample_X.copy()
nonnumeric.iloc[0, 0] = 'not-a-number'
malformed_nonnumeric_rejected = rejected(nonnumeric)
assert malformed_missing_rejected and malformed_extra_rejected and malformed_nonnumeric_rejected
artifact_replay_tests = pd.DataFrame([
    {'test': 'prediction_round_trip_atol_1e-10', 'passed': prediction_round_trip_atol_1e_10},
    {'test': 'recommendation_round_trip_atol_1e-10', 'passed': recommendation_round_trip_atol_1e_10},
    {'test': 'malformed_missing_rejected', 'passed': malformed_missing_rejected},
    {'test': 'malformed_extra_rejected', 'passed': malformed_extra_rejected},
    {'test': 'malformed_nonnumeric_rejected', 'passed': malformed_nonnumeric_rejected},
])
display(artifact_replay_tests)

## 最终自动测试汇总

In [ ]:
final_test_summary = pd.DataFrame([
    {'test': 'Excel炉次数=292', 'passed': len(df) == 292},
    {'test': '开发/锁定=248/44', 'passed': len(dev) == 248 and len(locked) == 44},
    {'test': '三个大时间折精确且无未来泄漏', 'passed': observed_windows == expected_windows},
    {'test': '完整19候选模型矩阵', 'passed': model_cv_summary['candidate'].nunique() == 19},
    {'test': '开发Champion在锁定审计前冻结', 'passed': experiment.frozen},
    {'test': '锁定审计覆盖19模型和44炉', 'passed': len(locked_audit) == 19 and len(locked) == 44},
    {'test': 'Conformal与GPR原生不确定性可用', 'passed': model_cv_summary['conformal_radius_90'].notna().all() and locked_audit['native_model_std_mean'].notna().sum() >= 2},
    {'test': '三优化器统一600预算', 'passed': set(optimizer_comparison['evaluations']) == {600}},
    {'test': '炉门次数保持整数', 'passed': bool((recommendation_summary[DOOR_COUNT_COL].dropna() % 1 == 0).all())},
    {'test': '安全门分项完整', 'passed': {'savings_pass','consensus_pass','feasibility_pass','boundary_pass','trust_region_pass','safety_pass'}.issubset(recommendation_summary.columns)},
    {'test': 'joblib预测和推荐可复现', 'passed': bool(artifact_replay_tests['passed'].all())},
    {'test': '离线节省免责声明存在', 'passed': '不等于工厂实际节省' in bundle['offline_savings_disclaimer']},
])
assert bool(final_test_summary['passed'].all()), final_test_summary[~final_test_summary['passed']]
display(final_test_summary)
print(f'开发阶段 Champion: {selected_before_lock}')
print(f'锁定集 RMSE 优胜者: {locked_rmse_winner}')
print(f'生产推荐模型: {production_recommendation_model}')
print(f"安全门结果: {production_recommendation.get('safety_pass', False)}")
print(bundle['offline_savings_disclaimer'])